# 1. Import and Hardware Setup

In [1]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import time
import copy

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [13]:
DATA_PATH = './Data'
SAVE_PATH = './Model'

# 2. Hyperparameters

In [14]:
BATCH_SIZE = 128
IMG_SIZE = 227
IMG_CHANNELS = 3
NUM_CLASSES = 101

EPOCHS = 30
LR = 1e-3
DROPOUT_RATE = 0.5


# 3. Dataset Preparation

In [15]:
stats = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [16]:
# Load the full train data
full_train_data = datasets.Food101(root=DATA_PATH, download=True, split='train', transform=train_transform)

# Split the full train data into train and validation data
train_size = int(0.8 * len(full_train_data))
val_size = len(full_train_data) - train_size

train_subset, val_subset = random_split(full_train_data, (train_size, val_size))

# Change transform von val_subset to test_transform
val_subset.dataset = copy.copy(full_train_data)
val_subset.dataset.transform = test_transform

# Load Test Dataset
test_dataset = datasets.Food101(root=DATA_PATH, download=True, split='test', transform=test_transform)

In [17]:
# Loader
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# 4. AlexNet Architecture

![AlexNet](AlexNet.png)

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, img_size, in_channels, num_classes, dropout_rate):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            # Conv 1: 227x227 -> 55x55
            nn.Conv2d(in_channels, 96, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),

            # MaxPool 1: 55x55 -> 27x27
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv 2: 27x27 -> 27x27
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),

            # MaxPool 2: 27x27 -> 13x13
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv 3: 13x13 -> 13x13
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv 4: 13x13 -> 13x13
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv 5: 13x13 -> 13x13
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # MaxPool 3: 13x13 -> 6x6
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(9216, 4096), #  6x6x256 = 9216 -> 4096
            nn.ReLU(inplace=True),

            nn.Dropout(dropout_rate),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        return logits

model = AlexNet(IMG_SIZE, IMG_CHANNELS, NUM_CLASSES, DROPOUT_RATE)

# 5. Training

In [ ]:
class EarlyStopping:
    """ Early stops the training if validation loss doesn't improve after 
        a given patience.
    """
    def __init__(self, patience=5, verbose=False, delta=0, save_path='best_checkpoint.pth', 
                                            trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.save_path = save_path
        self.trace_func = trace_func
        
        self.counter = 0
        self.best_score = None
        self.val_loss_min = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model):

        # 1. First epoch
        if self.best_score is None:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)

        # 2. If the validation loss is larger than the best score
        # -> Increase the counter
        elif val_loss > self.best_score - self.delta:
            self.counter += 1
            self.trace_func(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

        # 3. The Validation Loss reduced -> Save the model
        else:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        self.val_loss_min = val_loss
        torch.save(model.state_dict(), self.save_path)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)